In [ ]:
2e85f9b2978da8fd953ea9ddb384547ed674fd181f366211

# Building CAG Dataset from MSR cleaned data

In [6]:
import pandas as pd
df = pd.read_csv('MSR_data_cleaned.csv', nrows=10)

df.head(1).to_csv('MSR_data_cleaned_sample.csv', index=False)

In [7]:
df.head(1)

,Unnamed: 0,Access Gained,Attack Origin,Authentication Required,Availability,CVE ID,CVE Page,CWE ID,Complexity,Confidentiality,...,lang,lines_after,lines_before,parentID,patch,project,project_after,project_before,vul,vul_func_with_fix
0,0,NaN,Remote,Single system,Partial,CVE-2015-8467,https://www.cvedetails.com/cve/CVE-2015-8467/,CWE-264,Medium,Partial,...,C,NaN,NaN,a819d2b440aafa3138d95ff6e8b824da885a70e9,"@@ -1558,12 +1558,15 @@ static int samldb_chec...",samba,https://git.samba.org/?p=samba.git;a=blob;f=so...,https://git.samba.org/?p=samba.git;a=blob;f=so...,0,static bool check_rodc_critical_attribute(stru...


In [8]:
sample_codes = df[df["lang"] == "C"].head(5)['func_before'].tolist()

In [ ]:
from utils.cagutils import CAGDatasetBuilder
from utils.graphutils import CCodeCAGBuilder
import traceback

# Initialize builder
c_dataset_builder = CAGDatasetBuilder(CCodeCAGBuilder(use_cpp=False))



for idx, code in enumerate(sample_codes, 1):
    print(f"\n\n{'='*70}")
    print(f"Sample Code #{idx}")
    print('='*70)
    print(code)
    
    try:
        # Build CAGs
        cags = c_dataset_builder.builder.build_cag_from_code(code)
        
        # Display each function's CAG
        for func_name, cag in cags.items():
            
            # Show graph statistics
            print(f"\nCompression Statistics for {func_name}:")
            print(f"  Nodes: {len(cag.nodes)}")
            print(f"  Edges: {len(cag.edges)}")

    except Exception as e:
        print(f"Error processing code: {e}")
        traceback.print_exc()
    

c_dataset_builder.export_for_gnn('sample_c_cag_dataset')



Sample Code #1
static bool check_rodc_critical_attribute(struct ldb_message *msg)
{
	uint32_t schemaFlagsEx, searchFlags, rodc_filtered_flags;

	schemaFlagsEx = ldb_msg_find_attr_as_uint(msg, "schemaFlagsEx", 0);
	searchFlags = ldb_msg_find_attr_as_uint(msg, "searchFlags", 0);
	rodc_filtered_flags = (SEARCH_FLAG_RODC_ATTRIBUTE
			      | SEARCH_FLAG_CONFIDENTIAL);

	if ((schemaFlagsEx & SCHEMA_FLAG_ATTR_IS_CRITICAL) &&
		((searchFlags & rodc_filtered_flags) == rodc_filtered_flags)) {
		return true;
	} else {
		return false;
	}
}

Error processing code: <string>:1:13: before: check_rodc_critical_attribute


Sample Code #2
static int samldb_add_entry(struct samldb_ctx *ac)
{
	struct ldb_context *ldb;
	struct ldb_request *req;
	int ret;

	ldb = ldb_module_get_ctx(ac->module);

	ret = ldb_build_add_req(&req, ldb, ac,
				ac->msg,
				ac->req->controls,
				ac, samldb_add_entry_callback,
				ac->req);
	LDB_REQ_SET_LOCATION(req);
	if (ret != LDB_SUCCESS) {
		return ret;
	}

	return ldb_nex

Traceback (most recent call last):
  File "/tmp/ipykernel_249980/3875381015.py", line 17, in <module>
    cags = c_dataset_builder.builder.build_cag_from_code(code)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/vstor/courses/csds447/sxi219/utils/graphutils.py", line 524, in build_cag_from_code
    file_ast = self.parse_c_code(code)
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/vstor/courses/csds447/sxi219/utils/graphutils.py", line 127, in parse_c_code
    ast = self.parser.parse(code, filename=filename)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/easybuild_allnodes/software/Python-bundle-PyPI/2023.06-GCCcore-12.3.0/lib/python3.11/site-packages/pycparser/c_parser.py", line 147, in parse
    return self.cparser.parse(
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/easybuild_allnodes/software/Python-bundle-PyPI/2023.06-GCCcore-12.3.0/lib/python3.11/site-packages/pycparser/ply/yacc.py", line 331, in parse
    return self.p

{'graphs': [], 'labels': [], 'metadata': []}